# Vendor Comparison

Aggregates results across **all** tabular foundation model vendors and the shared
baselines, reading from the common `vendor_benchmark_results` Delta table that each
vendor notebook appends to via `common/evaluation.py`.

**Prerequisite:** run at least one vendor's notebooks first (e.g. `vendors/tabfm/`
and/or `vendors/tabpfn/`). Each writes rows tagged with its `vendor` name.

This notebook is vendor-agnostic — new vendors show up automatically once they log
results; no changes needed here.


In [ ]:
import sys
import os

# Make the repo-level common/ package importable.
# Adjust REPO_ROOT if your repo is checked out at a different workspace path.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
COMMON_PATH = os.path.join(REPO_ROOT, "common")
if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from config import CATALOG
SCHEMA = "default"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

RESULTS_TABLE = "vendor_benchmark_results"
results = spark.table(RESULTS_TABLE)
print(f"Total result rows: {results.count()}")
display(results.orderBy("task", "vendor", "model_name"))

## Best model per task

For classification we rank by ROC AUC (falling back to accuracy); for regression by R2.
This surfaces, per use case, whether a foundation model beats the tuned baselines.


In [ ]:
from pyspark.sql import functions as F, Window

cls = results.where(F.col("problem_type").like("%classification%"))
w_cls = Window.partitionBy("task").orderBy(
    F.col("roc_auc").desc_nulls_last(), F.col("accuracy").desc_nulls_last()
)
cls_ranked = (cls.withColumn("rank", F.row_number().over(w_cls))
                 .select("task", "rank", "vendor", "model_name",
                         "accuracy", "roc_auc", "f1"))
display(cls_ranked.orderBy("task", "rank"))

In [ ]:
reg = results.where(F.col("problem_type") == "regression")
w_reg = Window.partitionBy("task").orderBy(F.col("r2").desc_nulls_last())
reg_ranked = (reg.withColumn("rank", F.row_number().over(w_reg))
                 .select("task", "rank", "vendor", "model_name", "mae", "rmse", "r2"))
display(reg_ranked.orderBy("task", "rank"))

## Foundation models vs. baselines

Head-to-head: for each task, the best foundation-model vendor vs. the best baseline.


In [ ]:
fm = results.where(F.col("vendor") != "baseline")
bl = results.where(F.col("vendor") == "baseline")

def best_by(df, metric, higher_is_better=True):
    order = F.col(metric).desc_nulls_last() if higher_is_better else F.col(metric).asc_nulls_last()
    w = Window.partitionBy("task").orderBy(order)
    return df.withColumn("_r", F.row_number().over(w)).where(F.col("_r") == 1).drop("_r")

fm_best = best_by(fm.where(F.col("problem_type").like("%classification%")), "roc_auc")
bl_best = best_by(bl.where(F.col("problem_type").like("%classification%")), "roc_auc")
compare = (fm_best.select("task",
                          F.col("vendor").alias("fm_vendor"),
                          F.col("model_name").alias("fm_model"),
                          F.col("roc_auc").alias("fm_roc_auc"))
           .join(bl_best.select("task",
                                F.col("model_name").alias("baseline_model"),
                                F.col("roc_auc").alias("baseline_roc_auc")),
                 on="task", how="outer")
           .withColumn("fm_wins", F.col("fm_roc_auc") > F.col("baseline_roc_auc")))
display(compare.orderBy("task"))

## Notes

- **Task coverage differs by vendor.** TabPFN covers classification, regression,
  outlier detection, and forecasting; TabFM covers classification and regression only.
  A task only shows vendors that ran it.
- **License differs by vendor.** TabFM is non-commercial — see its README before
  acting on any comparison result for production.
- **Compute differs by vendor.** TabPFN runs on serverless CPU (hosted API); TabICL
  and TabFM need GPU. Because each vendor writes to the shared Delta table, they can
  be run on different clusters and compared here afterward.
